# MARV — layer x feature activation heatmaps (T4)

A different question than the first notebook: for **one prompt**, which
FFN features fire, and at which depth in the model? X-axis is feature
index, Y-axis is layer, color is cosine similarity between that layer's
hidden state and that layer's gate rows.

Compares a tool-triggering phrasing ("Call the weather tool") against a
plain phrasing of the same request ("What's the weather?") on the same
tool-tuned checkpoint, and looks at where they diverge most.

Runtime: **T4 GPU** (Runtime > Change runtime type > T4).

In [ ]:
!pip install -q transformers accelerate numpy matplotlib
!git clone -q https://github.com/thebnbrkr/marv.git /content/marv
%cd /content/marv
!pip install -q -e .

## Load the tool-tuned checkpoint and extract its vindex

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from marv.extract import extract

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

MODEL = "gvij/SmolLM2-135M-Function-Calling"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16).to(device).eval()

vindex = extract(model, model_name=MODEL)
print(vindex.num_layers, "layers,", vindex.gate[0].shape[0], "features/layer")

## A. Layer x feature heatmap for one prompt

Every feature at every layer for a single input -- no top-k reduction, this
is the full grid. `compute()` reuses the same contextual, last-token hidden
state approach as the first notebook (`marv.toolcall.hidden_states_at_layers`);
pass `baseline_prompt=` here too if you want to difference out a shared
template (useful for short/templated prompts -- see the first notebook's
"massive activation" note; full natural sentences like these two are usually
fine without it).

In [ ]:
from marv.layer_heatmap import compute, plot

TOOL_PROMPT = "Call the weather tool"
PLAIN_PROMPT = "What's the weather?"

hm_tool = compute(vindex, model, tok, TOOL_PROMPT, device=device)
hm_plain = compute(vindex, model, tok, PLAIN_PROMPT, device=device)
print("matrix shape (layers x features):", hm_tool.matrix.shape)

fig = plot(hm_tool, title=f"{MODEL}: {TOOL_PROMPT!r}")
fig

In [ ]:
fig = plot(hm_plain, title=f"{MODEL}: {PLAIN_PROMPT!r}")
fig

## Compare: where do the two prompts diverge most?

`difference()` subtracts one heatmap from the other cell-by-cell -- positive
(red) cells are features that fire more for the tool phrasing, negative
(blue) more for the plain phrasing.

In [ ]:
from marv.layer_heatmap import difference

d = difference(hm_tool, hm_plain)
fig = plot(d, title=f"difference: {TOOL_PROMPT!r} minus {PLAIN_PROMPT!r}", cmap="coolwarm", vmin=-0.4, vmax=0.4)
fig

In [ ]:
import numpy as np

row_max = np.abs(d.matrix).max(axis=1)
top_layers = np.argsort(-row_max)[:5]
print("layers with the largest tool-vs-plain divergence:")
for i in top_layers:
    layer = d.layers[i]
    feature = int(np.argmax(np.abs(d.matrix[i])))
    print(f"  L{layer}: |delta|={row_max[i]:.3f} at f{feature}")

## B. "Layer trace" -- with an honest caveat

A natural next ask is "show feature 134's activation across all layers."
That's not quite well-defined here: MARV's features are raw MLP neurons, one
gate row per layer -- feature index 134 at layer 3 and feature index 134 at
layer 18 are unrelated neurons that happen to share a column position, not a
persistent identity threading through the model (that *is* meaningful in
sparse-autoencoder-based interpretability, which builds an explicit shared
feature dictionary -- MARV doesn't have one).

What *is* well-defined and answers the same underlying question ("which
layers matter most for this input"): the strongest-firing feature *at each
layer*, tracked layer by layer. `peak_activation_trace()` gives you that.

In [ ]:
from marv.layer_heatmap import peak_activation_trace

peak_tool, feat_tool = peak_activation_trace(hm_tool)
peak_plain, feat_plain = peak_activation_trace(hm_plain)

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(hm_tool.layers, peak_tool, marker="o", label=TOOL_PROMPT)
ax.plot(hm_plain.layers, peak_plain, marker="o", label=PLAIN_PROMPT)
ax.set_xlabel("layer")
ax.set_ylabel("peak cosine similarity (strongest feature at that layer)")
ax.set_title("per-layer peak activation")
ax.legend()
fig.tight_layout()
fig

In [ ]:
# Raw column trace, if you want to check whether a *specific* feature index
# happens to stay active across nearby layers (see the caveat above for why
# this isn't "the same neuron" beyond coincidence).
from marv.layer_heatmap import layer_trace

# Pick a feature index that showed up as a peak somewhere interesting above.
example_feature = int(feat_tool[15])
print(f"feature index {example_feature} across all layers (tool prompt):")
print(layer_trace(hm_tool, example_feature))

## Next steps

- Swap `TOOL_PROMPT`/`PLAIN_PROMPT` for your own pair, or for a
  non-tool-calling behavior (e.g. an instruction-following phrasing vs. a
  plain statement) -- nothing here is tool-call-specific.
- Cross-reference the layers `difference()` flags as most divergent against
  the layers `marv.diff.most_changed()` found in the first notebook (15/20/22
  for this checkpoint) -- do the layers where fine-tuning wrote the most
  changes match the layers where tool vs. non-tool prompts diverge most at
  inference time?
- The polysemanticity heatmap from the first notebook (`marv.heatmap`) and
  this one answer different questions -- that one compares many words across
  categories at one layer, this one compares every feature across all layers
  for one prompt. Combining them (a category heatmap restricted to the
  layers this notebook flags as most divergent) is a natural next step.